# Build a Deep Research Agent over a Real Biomedical Corpus — Oracle AI Database + Claude

This notebook builds a **deep research agent**: the flagship use case for the
[`deepagents`](https://pypi.org/project/deepagents/) framework. 

The agent researches a **real
corpus of biomedical literature** — oncology abstracts from
[PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) — stored in the **Oracle AI Database**,
and is driven by **Anthropic's Claude** through Oracle `langchain-oci`'s *bring-your-own-model*
factory.


## The use case, in plain terms

Imagine a **research analyst** on an oncology tumor board. 

You hand them a broad clinical question —
*"What predicts survival and recurrence across cancers?"* — and a library of published studies. 

A good analyst doesn't answer off the cuff. They:

1. **Break the question into sub-topics** (prognostic factors, treatment response, recurrence, …).
2. **Search the literature** for evidence on each, one topic at a time.
3. **Keep organised notes** as they go, rather than holding everything in their head.
4. **Check** their draft against the sources before finalising.
5. **Deliver a tight, cited brief** a busy clinician can act on.

A plain chatbot can't do this reliably: it answers in one shot, loses the thread over a long task,
and — worst of all — **fabricates** when the evidence isn't there. A **deep agent** is built for
exactly this shape of work. Here we build that analyst, give it a real PubMed corpus (in the Oracle
AI Database), and watch it produce a grounded, **PubMed-ID-cited** brief.



> **Why real data?** The corpus is **40 real oncology abstracts** pulled from the PubMedQA dataset
> on Hugging Face (MIT-licensed). Every claim the agent makes can be traced to a real `PubMed:<id>`,
> so "grounded and cited" is verifiable, not a demo prop.

## The five powers of a deep agent

The agent demonstrates the five capabilities that make an agent *deep* rather than a
chat-with-tools loop:

1. **Planning** — it writes and tracks a to-do list before acting.
2. **A virtual filesystem** — it reads/writes working files outside the chat: scratch drafts for the
   current run, plus a `/memories/` folder that **persists to the Oracle AI Database** for long-term
   recall.
3. **Sub-agents** — it delegates focused retrieval to isolated helpers, then a critic verifies.
4. **Streaming** — you can watch the plan → act → observe loop unfold step by step.
5. **Durable memory** — conversation **checkpoints** and a long-term **`/memories/`** store, both
   persisted to the Oracle AI Database, so sessions resume after restarts and the agent recalls
   facts across threads.

## Architecture at a glance

| Component | Role | Where it runs |
|---|---|---|
| Knowledge base / vector store | grounds every answer | 🟢 **Local Docker** — Oracle AI Database Free; `langchain-oracledb` **`OracleVS`** |
| Vector index | fast semantic search | 🟢 **In-database** — Oracle AI Vector Search **HNSW** index |
| Embeddings (ingest + query) | text → vectors | 🟢 **Local** — HuggingFace `all-MiniLM-L6-v2` (CPU) |
| Retrieval | hybrid (vector + keyword) search | 🟢 **In-database** — HNSW vector index + Oracle Text |
| Reasoning / planning brain | drives the deep-agent loop | ☁️ **Claude** via `langchain-anthropic` |
| Agent harness | planning, files, sub-agents | `langchain-oci` → `deepagents` |
| Memory + checkpoints | durable agent state across runs/sessions | 🟢 **In-database** — `langgraph-oracledb` `OracleSaver` + `OracleStore` |

## Why the Oracle AI Database for a deep agent?

A deep agent touches a lot of state: 
- A knowledge base to ground answers, scratch files while it works
- Conversation checkpoints so a long run can resume
- And long-term memory it carries between sessions. 

The usual stack scatters those across a vector DB, a cache, a relational store, and an object store — four systems whose consistency, backups, and governance all have to line up.

The Oracle AI Database collapses that into **one backend**:

- **Retrieval** — `OracleVS` (from `langchain-oracledb`) stores embeddings in the native `VECTOR`
  type with an **HNSW** index, and an Oracle Text index adds keyword search — so a single
  in-database call does **hybrid** (semantic + keyword) retrieval, with no separate vector database.
- **Checkpoints** — `OracleSaver` (from `langgraph-oracledb`) persists the agent's per-`thread_id`
  state transactionally, so a long research session is durable and resumable.
- **Long-term memory** — `OracleStore` backs the agent's `/memories/` folder, persisting facts
  across sessions and threads.
- **One system of record** — vectors sit *next to* operational data, behind one connection pool,
  with the backups, HA, and governance a regulated (e.g. clinical) workload already needs. The data
  never leaves the database; only the reasoning is hosted.

Because the model is **bring-your-own** (`model=ChatAnthropic(...)`), the reasoning provider and the
data substrate are independent choices. This notebook uses all three Oracle integrations together —
`langchain-oracledb` (`OracleVS`), `langchain-oci` (the deepagents factory), and `langgraph-oracledb`
(`OracleSaver` / `OracleStore`).

## How the agent stores memory: two substrates, one filesystem

This agent keeps its working memory in **two substrates at once — a virtual filesystem and the
Oracle AI Database — and they are wired together**, not used in isolation. That link is the key to
the whole design, so it's worth stating up front.

**1 · The virtual filesystem (what the *agent* sees).** The agent never talks to Oracle directly
for memory. It reads and writes through deepagents' file tools (`write_file`, `read_file`, `ls`,
`edit_file`) against a single POSIX-like path namespace — e.g. `/research_request.md`,
`/final_report.md`, `/memories/board_prefs.md`. This is how it parks long artifacts (drafts, the
saved task, durable notes) *outside* the chat transcript so the context window stays small.

**2 · The Oracle AI Database (where the bytes actually live).** Behind that one filesystem, storage
is split by path with a deepagents `CompositeBackend`, so **every file the agent writes ends up in
Oracle** — just via two different routes:

| Filesystem path | Backend it routes to | Where it lands in Oracle | Lifetime |
|---|---|---|---|
| `/memories/**` | `StoreBackend` → `OracleStore` | First-class **rows** in the store tables — addressable by key, queryable with `search()` | **Cross-thread, cross-session** (survives fresh processes) |
| everything else (`/research_request.md`, `/final_report.md`, scratch) | `StateBackend` | Inside the LangGraph run **state**, which `OracleSaver` snapshots as the per-`thread_id` **checkpoint** | **Per-thread** (durable & resumable for that thread) |

So the bridge between "the filesystem" and "the database" is the **backend**:

- A write under `/memories/...` is a *direct* filesystem→database mapping — `StoreBackend` turns it
  into an `OracleStore` row you can read from any thread (proven in Part 5.2: save on one thread,
  recall on a brand-new one).
- A write to any other path goes into the agent's in-run state via `StateBackend`. That's fast (no
  DB round-trip per edit) but **not lost**: because we attach `OracleSaver`, the whole state —
  scratch files included — is checkpointed into Oracle after each step, keyed by `thread_id`, so the
  run can pause, resume, and survive a restart.

In short: **`/memories/` is the agent's durable, queryable long-term store (rows in Oracle);
everything else is fast scratch that *still* reaches Oracle, transitively, inside the per-thread
checkpoint.** One filesystem to the model, one database underneath, two persistence guarantees. We
build this backend in Part 3.2 and exercise both halves in Part 5.2.

## Key terms (read once, refer back as needed)

| Term | One-line meaning |
|---|---|
| **Deep agent** | An LLM given a *planning tool*, a *virtual filesystem*, and the ability to spawn *sub-agents* — scaffolding for long, multi-step work. |
| **Tool** | A function the LLM can choose to call; the framework runs it and feeds the result back. |
| **Sub-agent** | A scoped agent (own prompt) the orchestrator delegates a sub-task to; runs in an isolated context and returns a concise result. |
| **Context isolation** | Keeping a sub-task's bulky data inside a sub-agent so the orchestrator's context stays focused on planning and synthesis. |
| **HNSW** | *Hierarchical Navigable Small World* — a graph vector index for fast approximate nearest-neighbour search. |
| **Checkpoint** | A saved snapshot of the agent's per-`thread_id` state, so a run can resume after a restart. |
| **Bring-your-own-model** | Passing `model=` so any LangChain chat model drives the harness; OCI inference auth is bypassed. |

--------

# Part 1 · Stand up the Oracle AI Database

We run Oracle AI Database **Free** locally in Docker — no cloud account, no credentials leave your
machine.

## 1.1 · Prerequisites

**1. Oracle AI Database Free in Docker** (the [gvenzl](https://github.com/gvenzl/oci-oracle-free)
image, which includes Oracle AI Vector Search):

```bash
docker run -d --name oracle-free -p 1521:1521 \
  -e ORACLE_PASSWORD=Welcome12345 \
  gvenzl/oracle-free:latest
# wait until: docker logs oracle-free | grep "DATABASE IS READY TO USE"
```

**2. Python 3.11–3.13** (the `deepagents` support window).

**3. An Anthropic API key** — in `samples/11-deepagents/.env` as `ANTHROPIC_API_KEY=sk-ant-...`
(gitignored), or you'll be prompted.

## 1.2 · Install dependencies

In [1]:
# langchain-oci (BYO-model deepagents factory + Oracle datastores) from local source:
%pip install -q -e ../../libs/oci

# Oracle vector store + driver + LangGraph persistence:
%pip install -q langchain-oracledb langgraph-oracledb oracledb langchain-community

# Local embeddings, the Claude chat model, the real deepagents package, and the dataset library:
%pip install -q langchain-huggingface sentence-transformers langchain-anthropic "deepagents>=0.6" datasets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1.3 · Configuration

Everything is parameterised so you can point at Autonomous Database later by changing only the connection details.

In [2]:
import os

# --- Local Oracle AI Database (Docker) ---
ORACLE_DSN        = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")  # the PDB (app data)
ORACLE_CDB_DSN    = os.environ.get("ORACLE_CDB_DSN", "localhost:1521/FREE")  # the CDB root (SGA params)
ORACLE_ADMIN_USER = os.environ.get("ORACLE_ADMIN_USER", "SYSTEM")
ORACLE_ADMIN_PWD  = os.environ.get("ORACLE_ADMIN_PWD", "Welcome12345")       # = ORACLE_PASSWORD
APP_USER          = os.environ.get("ORACLE_APP_USER", "DEEPAGENTS")
APP_PWD           = os.environ.get("ORACLE_APP_PWD", "Welcome12345")
TABLE_NAME        = os.environ.get("ORACLE_TABLE", "PUBMED_DOCS")

# --- Local embeddings (HuggingFace, CPU) ---
HF_EMBED_MODEL    = os.environ.get("HF_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
HF_EMBED_DEVICE   = os.environ.get("HF_EMBED_DEVICE", "cpu")

# --- Reasoning brain (Claude) ---
CLAUDE_MODEL      = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")

print("DB        :", ORACLE_DSN, "| app user:", APP_USER, "| table:", TABLE_NAME)
print("Embeddings:", HF_EMBED_MODEL, "(local,", HF_EMBED_DEVICE + ")")
print("Brain     :", CLAUDE_MODEL, "(Anthropic)")

DB        : localhost:1521/FREEPDB1 | app user: DEEPAGENTS | table: PUBMED_DOCS
Embeddings: sentence-transformers/all-MiniLM-L6-v2 (local, cpu)
Brain     : claude-sonnet-4-6 (Anthropic)


## 1.4 · Load the Anthropic API key

Reads `.env` next to the notebook first (works headless); otherwise prompts.

In [3]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("ANTHROPIC_API_KEY"):
    load_dotenv(os.path.join(os.getcwd(), ".env"))
if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
print("Anthropic API key loaded.")

Anthropic API key loaded.


## 1.5 · Verify the database is reachable

Before doing any real work, confirm the container is up and accepting connections. The cell below
opens a connection to the running Oracle instance with `python-oracledb` (thin mode — no Oracle
client install needed), connecting as the admin user to the `FREEPDB1` PDB, and runs a trivial
`select banner from v$version`. If it prints the database version, then connectivity, credentials,
and the DSN are all good. The `with` block closes the connection automatically. A failure here —
rather than deep inside ingestion later — tells you immediately that the Docker container isn't
ready or the connection details are wrong.

In [4]:
import oracledb

with oracledb.connect(user=ORACLE_ADMIN_USER, password=ORACLE_ADMIN_PWD, dsn=ORACLE_DSN) as con:
    (banner,) = con.cursor().execute("select banner from v$version").fetchone()
print("Connected:", banner)

Connected: Oracle AI Database 26ai Free Release 23.26.0.0.0 - Develop, Learn, and Run for Free


## 1.6 · Create a dedicated application user

Good hygiene: the agent connects as a low-privilege `DEEPAGENTS` user, not `SYSTEM`.

In [5]:
import oracledb

with oracledb.connect(user=ORACLE_ADMIN_USER, password=ORACLE_ADMIN_PWD, dsn=ORACLE_DSN) as con:
    cur = con.cursor()
    cur.execute("select count(*) from all_users where username = :u", u=APP_USER)
    if cur.fetchone()[0] == 0:
        cur.execute(f'create user {APP_USER} identified by "{APP_PWD}" quota unlimited on users')
        cur.execute(f"grant db_developer_role to {APP_USER}")
        print(f"Created application user {APP_USER}.")
    else:
        print(f"Application user {APP_USER} already exists.")

Application user DEEPAGENTS already exists.


## 1.7 · Enable Oracle AI Vector Search HNSW memory (`VECTOR_MEMORY_SIZE`)

HNSW vector indexes live in a dedicated SGA pool, `VECTOR_MEMORY_SIZE`, which defaults to **0** on
Oracle Free. It's an **instance-level** parameter, so it must be set at the **CDB root** (`FREE`),
not the PDB — and it requires a one-time restart.

This cell checks the running value; if it's 0, it sets the parameter and tells you to restart the
container once. If it's already non-zero, you're ready for HNSW.

In [6]:
import oracledb

with oracledb.connect(user=ORACLE_ADMIN_USER, password=ORACLE_ADMIN_PWD, dsn=ORACLE_CDB_DSN) as cdb:
    cur = cdb.cursor()
    running = int(cur.execute("select value from v$parameter where name='vector_memory_size'").fetchone()[0])
    if running > 0:
        print(f"VECTOR_MEMORY_SIZE = {running/1024/1024:.0f} MB — HNSW is ready.")
    else:
        cur.execute("alter system set vector_memory_size=512M scope=spfile")
        print("VECTOR_MEMORY_SIZE set to 512M in the SPFILE (at the CDB root).")
        print("➡  Restart the database ONCE, then re-run Part 1 from here:")
        print("     docker restart oracle-free")
        raise SystemExit("Restart required to allocate the vector memory pool, then re-run.")

VECTOR_MEMORY_SIZE = 512 MB — HNSW is ready.


-------

# Part 2 · Load and index the real PubMed corpus

We pull real oncology abstracts from PubMedQA, embed them locally, and index them in Oracle with an
**HNSW** vector index plus an Oracle Text index for hybrid search.

## 2.1 · Build the local embedding model

HuggingFace `all-MiniLM-L6-v2` runs on CPU and never leaves your machine.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=HF_EMBED_MODEL,
    model_kwargs={"device": HF_EMBED_DEVICE},
)
print("Embedding dimension:", len(embedding_model.embed_query("hello")))

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_eco/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2550.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


## 2.2 · The knowledge base — real oncology abstracts from PubMedQA

[PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) (MIT) pairs biomedical research
questions with the **abstract** of the paper that answers them, plus a `pubid`. We take a coherent
**oncology** slice so the corpus hangs together as one research domain — and every document carries
a real `PubMed:<id>` the agent can cite.

In [8]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from datasets import load_dataset

ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

ONCOLOGY = ("cancer", "tumor", "tumour", "carcinoma", "oncolog", "metasta", "chemotherap")
CORPUS = []
for row in ds:
    ctx = row["context"]
    body = " ".join(ctx.get("contexts", [])) if isinstance(ctx, dict) else str(ctx)
    if any(k in (row["question"] + " " + body).lower() for k in ONCOLOGY):
        CORPUS.append({
            "id": str(row["pubid"]),
            "title": row["question"].strip(),
            "content": body.strip(),
            "source": f"PubMed:{row['pubid']}",
        })
    if len(CORPUS) >= 40:
        break

print(f"Loaded {len(CORPUS)} real oncology abstracts from PubMedQA.")

Loaded 40 real oncology abstracts from PubMedQA.


A quick look at one real document (this is published literature, not synthetic copy):

In [9]:
doc = CORPUS[0]
print("Doc ID :", doc["id"])
print("Title  :", doc["title"])
print("Abstract:", doc["content"][:400], "...")

Doc ID : 25957366
Title  : Prompting Primary Care Providers about Increased Patient Risk As a Result of Family History: Does It Work?
Abstract: Electronic health records have the potential to facilitate family history use by primary care physicians (PCPs) to provide personalized care. The objective of this study was to determine whether automated, at-the-visit tailored prompts about family history risk change PCP behavior. Automated, tailored prompts highlighting familial risk for heart disease, stroke, diabetes, and breast, colorectal, o ...


And the first few rows of the whole corpus as a table — `id`, citation `source`, the research
`title`, and a truncated `content` — so you can see the shape of what we're about to embed and
index:

In [ ]:
# A quick tabular view (a "dataframe") of the first few abstracts now sitting in CORPUS.
import pandas as pd

preview = pd.DataFrame(CORPUS)[["id", "source", "title", "content"]].head()
preview["content"] = preview["content"].str.slice(0, 100) + "…"  # truncate long abstracts for display
preview

## 2.3 · Ingest into the Oracle AI Database

`ADB` is `langchain-oci`'s datastore adapter; under the hood it's `langchain-oracledb`'s `OracleVS`
vector store. We reset the table, build the store, and write the abstracts (each stored as one
passage — abstracts are short).

**Step 1 — reset the table** (the vector column's dimension is tied to the embedding model, so we start clean):

In [11]:
import oracledb

with oracledb.connect(user=APP_USER, password=APP_PWD, dsn=ORACLE_DSN) as con:
    cur = con.cursor()
    try:
        cur.execute(f"DROP TABLE {TABLE_NAME} PURGE")
        print(f"· reset existing table {TABLE_NAME}")
    except oracledb.DatabaseError as exc:
        if exc.args[0].code != 942:  # ORA-00942: table does not exist
            raise
    con.commit()

· reset existing table PUBMED_DOCS


**Step 2 — build the `ADB` datastore** (this is the `OracleVS`-backed vector store):

> **💡 One database for relational *and* vector data.** The `ADB` store below writes each abstract's
> **text**, its **metadata** (PubMed id, title, source), **and** its embedding **`VECTOR`** into the
> *same* Oracle table. Oracle AI Database treats vectors as a native column type sitting right
> alongside ordinary relational columns — so you get semantic search *and* SQL filtering/joins
> against operational data in one place. There's no separate, specialized vector database to deploy,
> secure, back up, and keep in sync: the embeddings live next to the data they describe, behind one
> connection.

In [12]:
from langchain_oci.datastores import ADB

store = ADB(
    dsn=ORACLE_DSN,
    user=APP_USER,
    password=APP_PWD,
    table_name=TABLE_NAME,
    datastore_description=(
        "Biomedical research abstracts (oncology) from PubMed: prognosis, treatment "
        "response, recurrence, survival, screening, and clinical outcomes."
    ),
    chunk_on_write=False,  # abstracts are short -> store each as a single passage
)
store.connect(embedding_model)
print("ADB datastore connected to", f"{APP_USER}.{TABLE_NAME}")

ADB datastore connected to DEEPAGENTS.PUBMED_DOCS


**Step 3 — write the abstracts** (embeddings are computed locally on ingest):

In [13]:
ingested = store.bulk_insert(CORPUS, embeddings=[[] for _ in CORPUS])
print(f"Ingested {ingested} abstracts. Store stats:", store.stats()["document_count"], "documents")

Ingested 40 abstracts. Store stats: 40 documents


## 2.4 · Index for fast hybrid search — HNSW + Oracle Text

Two indexes make retrieval both fast and precise:
- an **HNSW** vector index for approximate nearest-neighbour *semantic* search, and
- an **Oracle Text** index for *keyword* search.

Together they let the `search` tool fuse semantic + keyword results (hybrid retrieval) in one
in-database call.

**The HNSW vector index** (`langchain-oracledb`'s `create_index`). Requires the `VECTOR_MEMORY_SIZE` pool from Part 1.7:

In [14]:
from langchain_oracledb.vectorstores.oraclevs import create_index

try:
    create_index(
        store.vectorstore.client,
        store.vectorstore,
        params={"idx_name": f"{TABLE_NAME}_HNSW", "idx_type": "HNSW"},
    )
    print("✓ HNSW vector index created.")
except Exception as exc:
    if "ORA-00955" in str(exc):
        print("· HNSW vector index already exists.")
    elif "ORA-51962" in str(exc):
        print("✗ Vector memory pool is 0 — re-run Part 1.7 and restart the DB to enable HNSW.")
    else:
        raise

✓ HNSW vector index created.


**The Oracle Text index** (enables keyword search, so `search` runs *hybrid* retrieval):

In [15]:
cur = store.vectorstore.client.cursor()
try:
    cur.execute(f'CREATE SEARCH INDEX {TABLE_NAME}_TXT_IDX ON {TABLE_NAME}("TEXT")')
    print("✓ Oracle Text index created — hybrid (vector + keyword) search enabled.")
except oracledb.DatabaseError as exc:
    if "ORA-00955" in str(exc):
        print("· Oracle Text index already exists.")
    else:
        raise
finally:
    cur.close()

✓ Oracle Text index created — hybrid (vector + keyword) search enabled.


**Confirm the indexes** — note `*_HNSW` is reported as a `VECTOR` index, and the store engine is `langchain-oracledb`'s `OracleVS`:

In [16]:
cur = store.vectorstore.client.cursor()
cur.execute("select index_name, index_type from user_indexes where table_name = :t order by 1", t=TABLE_NAME)
for name, itype in cur.fetchall():
    print(f"  {name:24} {itype}")
cur.close()
vs = store.vectorstore
print("\nVector store engine:", f"{type(vs).__module__}.{type(vs).__name__}")

  IDX_PUBMED_DOCS_MID      FUNCTION-BASED NORMAL
  PUBMED_DOCS_HNSW         VECTOR
  PUBMED_DOCS_TXT_IDX      DOMAIN
  SYS_C008841              NORMAL
  SYS_IL0000073018C00002$$ LOB
  SYS_IL0000073018C00003$$ LOB
  SYS_IL0000073018C00004$$ LOB

Vector store engine: langchain_oracledb.vectorstores.oraclevs.OracleVS


## 2.5 · Sanity-check retrieval

A grounded query should return relevant abstracts straight from Oracle. `create_deepagents_agent(datastores=...)` will build the `search` / `get_document` / `stats` tools from this store automatically.

In [17]:
hits = store.search_documents("which factors predict cancer survival and recurrence", top_k=3)
print(f"Retrieved {len(hits)} grounded abstract(s) from Oracle AI Database. Top match:\n")
print(f"[PubMed:{hits[0].metadata.get('id')}]", hits[0].page_content[:280], "...")

Retrieved 3 grounded abstract(s) from Oracle AI Database. Top match:

[PubMed:16968876] The aim of this prognostic factor analysis was to investigate if a patient's self-reported health-related quality of life (HRQOL) provided independent prognostic information for survival in non-small cell lung cancer (NSCLC) patients. Pretreatment HRQOL was measured in 391 advanc ...


# Part 3 · Build the deep research agent

Now the agent itself: a reflection tool, two sub-agents, an Oracle-backed memory layer, and the
bring-your-own-Claude assembly.

## 3.1 · The reflection tool and the sub-agents

Three ingredients make the research loop disciplined. We define them one at a time.

**`think_tool`** — a no-op *strategic pause*. After each search the agent states what it found, what's missing, and whether to keep digging. Forcing reflection measurably improves multi-step research.

In [18]:
from langchain_core.tools import tool

@tool
def think_tool(reflection: str) -> str:
    """Record a strategic reflection: what evidence was found, what gaps remain, and whether to
    keep researching or move on. Call this after each search to stay focused."""
    return f"Reflection recorded: {reflection}"

**`kb-researcher`** — a sub-agent that researches *one* sub-topic against PubMed and returns distilled, cited findings. It **omits `tools`**, so it inherits the orchestrator's Oracle search tools; the orchestrator delegates each sub-topic to a *fresh* instance, keeping noisy retrieved text out of the orchestrator's context (**context isolation**).

In [19]:
research_subagent = {
    "name": "kb-researcher",
    "description": (
        "Researches ONE narrow sub-topic against the PubMed oncology corpus and returns concise, "
        "cited findings. Delegate a single, focused sub-topic per call."
    ),
    "system_prompt": (
        "You are a focused biomedical research specialist. Use the `search` tool to find evidence "
        "in the PubMed corpus, `get_document` to pull a full abstract when a snippet is not enough, "
        "and call `think_tool` after searching to assess what is still missing. Return a section "
        "titled '## Key Findings' with 3-5 concise bullets, each citing a PubMed ID inline like "
        "[PubMed:25957366]. Use ONLY retrieved content; if the corpus has nothing relevant, say so "
        "plainly. No preamble, no conclusion."
    ),
    # tools omitted -> inherits the orchestrator's tools (Oracle search + think_tool)
}

**`critic`** — a sub-agent that re-checks the draft's claims against the corpus and flags anything unsupported: a built-in guard against hallucination.

In [20]:
critique_subagent = {
    "name": "critic",
    "description": (
        "Verifies a draft answer's claims against the PubMed corpus and lists unsupported claims, "
        "missing caveats, or citation errors. Use before finalizing."
    ),
    "system_prompt": (
        "You are a skeptical biomedical reviewer. For each claim in the draft you are given, use "
        "`search` and `get_document` to confirm it is supported by the corpus. Return a short "
        "bullet list of: unsupported or overstated claims, missing caveats, and citation errors. "
        "If everything checks out, say so briefly. Be concrete and concise."
    ),
    # tools omitted -> inherits the orchestrator's Oracle search tools
}

## 3.2 · Make Oracle the agent's memory — checkpoints + long-term store

Retrieval is only half of what a production deep agent needs from a database; the other half is
**durable state**. We add two Oracle-backed layers, one at a time:

- **Checkpointing** (`OracleSaver`) — every step of a run is saved per `thread_id`, so a long
  session can pause, resume, and survive restarts.
- **Long-term memory** (`OracleStore`) — a `/memories/` folder, backed by a LangGraph store, that
  **persists across sessions**.

We route **only** `/memories/` to Oracle via a `CompositeBackend`; scratch drafts stay in fast
in-run `StateBackend`.

**Open two connections** to the same Oracle AI Database used for retrieval:

In [21]:
import oracledb

saver_conn = oracledb.connect(user=APP_USER, password=APP_PWD, dsn=ORACLE_DSN)
store_conn = oracledb.connect(user=APP_USER, password=APP_PWD, dsn=ORACLE_DSN)
print("Opened checkpoint + store connections.")

Opened checkpoint + store connections.


**Checkpointer** — durable per-thread state (`setup()` is idempotent; it creates the checkpoint tables):

In [22]:
from langgraph_oracledb.checkpoint.oracle import OracleSaver

checkpointer = OracleSaver(saver_conn)
checkpointer.setup()
print("OracleSaver ready (checkpoint tables ensured).")

OracleSaver ready (checkpoint tables ensured).


**Long-term store** — the durable `/memories/` backend:

In [23]:
from langgraph_oracledb.store.oracle import OracleStore

memory_store = OracleStore(store_conn)
memory_store.setup()
print("OracleStore ready (store tables ensured).")

OracleStore ready (store tables ensured).


**Compose the filesystem backend** — `/memories/` → Oracle; everything else → fast in-run state:

**What each backend does — and how they combine.** deepagents' filesystem is *pluggable*: a
**backend** decides where the agent's files physically live. We use three classes together:

- **`StateBackend`** — stores files in the agent's in-run LangGraph **state** (the `files` channel).
  Fast and ephemeral: no database round-trip while the agent drafts. It's the right home for scratch
  artifacts like `/research_request.md` and `/final_report.md`. (With our `OracleSaver` attached,
  this state is still checkpointed to Oracle per `thread_id` — so it's durable *for that thread*,
  just not a standalone, queryable record.)
- **`StoreBackend`** — adapts a LangGraph **`BaseStore`** (here `OracleStore`) into the filesystem.
  Files written through it become **rows in Oracle**, addressable by key and **persistent across
  threads and sessions**. We point it at the `/memories/` folder for the agent's long-term memory.
- **`CompositeBackend`** — the **router**. It takes a default backend plus a *prefix → backend* map
  and sends each path to the right place. Here: `/memories/**` → `StoreBackend` (Oracle); everything
  else → `StateBackend` (fast scratch).

Net effect for the agent: it sees **one** filesystem, but writes under `/memories/` become durable
Oracle records while all other writes stay fast in-run scratch (still snapshotted via the
checkpointer). This is the deepagents-recommended pattern — **pay for durability only where it
matters.**

In [24]:
from deepagents.backends.composite import CompositeBackend
from deepagents.backends.state import StateBackend
from deepagents.backends.store import StoreBackend

backend = CompositeBackend(
    StateBackend(),
    {"/memories/": StoreBackend(
        store=memory_store,
        namespace=lambda ctx: ("oncology", "memories"),  # in prod, scope by user / org
    )},
)
print("CompositeBackend ready — /memories/ persists to Oracle, scratch stays in state.")

CompositeBackend ready — /memories/ persists to Oracle, scratch stays in state.


## 3.3 · Assemble the deep agent — bring-your-own Claude

The assembly in three parts: the model, the orchestrator's instructions, then the factory call.

**The model** — this is the entire bring-your-own-model story. `model=ChatAnthropic(...)` hands the factory any LangChain chat model and bypasses OCI inference auth; the Oracle harness is driven by Claude.

In [25]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model=CLAUDE_MODEL,
    temperature=0,     # deterministic planning/synthesis
    max_tokens=8000,
    timeout=300,
)

**The orchestrator's instructions** — the classic deep-research workflow: *plan → save → delegate → verify → synthesize*.

In [26]:
RESEARCH_LEAD_PROMPT = (
    "You are the research lead on an oncology evidence team. Produce rigorous, fully-grounded "
    "briefs.\n\n"
    "Follow this workflow for every request:\n"
    "1. PLAN: call `write_todos` to break the question into focused sub-topics.\n"
    "2. SAVE: write the user's exact question to `/research_request.md` with your file tools.\n"
    "3. RESEARCH: delegate EACH sub-topic to the `kb-researcher` subagent via the `task` tool — "
    "one sub-topic per call. Do NOT search the corpus yourself; the researchers do that.\n"
    "4. VERIFY: assemble a draft, then send it to the `critic` subagent and address its findings.\n"
    "5. SYNTHESIZE: write the final brief to `/final_report.md`, then return it as your answer.\n\n"
    "Rules: every factual claim MUST cite a PubMed ID inline like [PubMed:25957366]. If the corpus "
    "does not support a claim, say so rather than guessing. Keep the final brief tight and "
    "clinician-ready."
)

**The factory call** — `datastores=` wires the Oracle search tools *and* keeps the full deep-agent harness; `checkpointer` / `store` / `backend` make Oracle the memory layer; sub-agents inherit the search tools.

**Every argument in the factory call below, in detail:**

- **`model=llm`** — bring-your-own-model. The factory drives the deep-agent harness with this
  `ChatAnthropic` (Claude) instance and **bypasses OCI inference auth** entirely; `model_id` and OCI
  credentials are ignored.
- **`datastores={"pubmed": store}`** — registers the Oracle `ADB`/`OracleVS` store. The factory
  auto-builds the `search`, `get_document`, and `stats` tools from it, so the agent (and its
  sub-agents) can retrieve from PubMed. The dict key (`"pubmed"`) names the store for routing when
  you register more than one.
- **`embedding_model=embedding_model`** — the local HuggingFace embedder used to embed *queries* at
  search time. Supplying it keeps retrieval fully local and avoids any OCI embedding calls.
- **`tools=[think_tool]`** — extra custom tools for the **orchestrator**. We add only `think_tool`;
  the system prompt tells the orchestrator to *delegate* searching rather than do it itself.
- **`subagents=[research_subagent, critique_subagent]`** — the sub-agent specs. Passing `subagents=`
  enables the built-in `task` delegation tool. Because each spec omits `tools`, the sub-agents
  inherit the orchestrator's Oracle search tools.
- **`checkpointer=checkpointer`** — the `OracleSaver`. Persists the full run state per `thread_id`
  after each step, making a long session durable and resumable (demonstrated in Part 5.2a).
- **`store=memory_store`** — the `OracleStore`. The LangGraph long-term store that the `/memories/`
  backend writes to; persists across threads and sessions (Part 5.2b).
- **`backend=backend`** — the `CompositeBackend` from 3.2 that routes `/memories/` → Oracle and
  everything else → fast in-run state.
- **`system_prompt=RESEARCH_LEAD_PROMPT`** — the orchestrator's instructions: the
  *plan → save → delegate → verify → synthesize* workflow with mandatory PubMed-ID citations.

In [27]:
from langchain_oci import create_deepagents_agent

agent = create_deepagents_agent(
    model=llm,                                       # <-- bring-your-own Claude
    datastores={"pubmed": store},                    # one-liner: Oracle retrieval tools + deep harness
    embedding_model=embedding_model,                 # local HF embeddings -> no OCI calls
    tools=[think_tool],                              # orchestrator delegates; prompt tells it not to search
    subagents=[research_subagent, critique_subagent],# sub-agents inherit the Oracle search tools
    checkpointer=checkpointer,                       # durable thread state in Oracle (OracleSaver)
    store=memory_store,                              # long-term memory store in Oracle (OracleStore)
    backend=backend,                                 # route /memories/ -> Oracle
    system_prompt=RESEARCH_LEAD_PROMPT,
)

In [28]:
print("Agent type:", type(agent).__name__)
print("Driven by the Claude model we passed in:", agent._oci_llm is llm)

Agent type: CompiledStateGraph
Driven by the Claude model we passed in: True


## 3.4 · Display helpers

Small helpers to render what the agent produces from its state dict (`messages`, `todos`, `files`) and its delegations.

In [29]:
from IPython.display import Markdown, display


def show_flow(state):
    print(" -> ".join(type(m).__name__ for m in state["messages"]))


def show_todos(state):
    todos = state.get("todos") or []
    if not todos:
        print("(no plan recorded)"); return
    mark = {"completed": "done", "in_progress": "in progress", "pending": "pending"}
    rows = ["| # | Task | Status |", "|---|------|--------|"]
    for i, t in enumerate(todos, 1):
        rows.append(f"| {i} | {t.get('content','')} | {mark.get(t.get('status'), t.get('status',''))} |")
    display(Markdown("\n".join(rows)))


def show_files(state):
    files = state.get("files") or {}
    if not files:
        print("(no files written)"); return
    for path, blob in files.items():
        content = blob.get("content") if isinstance(blob, dict) else blob
        print(f"FILE {path}  ({len(content)} chars)")
        display(Markdown(content))


def show_delegations(state):
    dels = [tc for m in state["messages"]
            for tc in (getattr(m, "tool_calls", None) or []) if tc["name"] == "task"]
    print(f"{len(dels)} delegation(s) via the built-in `task` tool:")
    for tc in dels:
        a = tc["args"]
        print(f"  -> [{a.get('subagent_type')}] {str(a.get('description',''))[:90]}")

# Part 4 · Run the research and inspect the powers

## 4.1 · Run the deep research task

A broad clinical question that *demands* depth. The flow will be long — the orchestrator plans,
delegates several sub-topics to `kb-researcher`, runs the `critic`, and synthesizes. This makes
**live Claude API calls** and takes a few minutes; that latency is the deep-agent loop doing real
work. We pass a `thread_id` so the run is checkpointed to Oracle.

In [30]:
import uuid
from langchain_core.messages import HumanMessage

RESEARCH_TASK = (
    "Produce a one-page evidence brief for an oncology tumor board on what predicts survival and "
    "recurrence across cancers. Cover: (1) the strongest prognostic factors, (2) how treatment "
    "choice affects outcomes, (3) where the evidence is mixed or uncertain, and (4) a practical "
    "recommendation for follow-up. Research each angle against the PubMed corpus, verify the draft "
    "with the critic, and cite PubMed IDs throughout."
)

In [31]:
# Unique per run so re-executing starts a fresh checkpointed thread (idempotent).
RESEARCH_CONFIG = {"configurable": {"thread_id": f"oncology-brief-{uuid.uuid4().hex[:8]}"}}
state = agent.invoke(
    {"messages": [HumanMessage(content=RESEARCH_TASK)]},
    config=RESEARCH_CONFIG,
)
show_flow(state)

KeyboardInterrupt: 

## 4.2 · Power #1 — Planning

The to-do list the agent built and worked through, before writing a word of the brief.

In [ ]:
show_todos(state)

| # | Task | Status |
|---|------|--------|
| 1 | Save research request to /research_request.md | done |
| 2 | Research sub-topic 1: Strongest prognostic factors for survival and recurrence across cancers | done |
| 3 | Research sub-topic 2: How treatment choice affects survival and recurrence outcomes | done |
| 4 | Research sub-topic 3: Mixed or uncertain evidence in cancer prognosis and recurrence prediction | done |
| 5 | Research sub-topic 4: Evidence-based recommendations for post-treatment follow-up | done |
| 6 | Assemble draft brief and send to critic subagent for verification | done |
| 7 | Write final verified brief to /final_report.md and return to user | done |

## 4.3 · Power #2 — The virtual filesystem

The documents the agent drafted in its workspace. `/research_request.md` is the saved task;
`/final_report.md` is the brief it composed. These scratch files live in fast **in-run state** (the
default `StateBackend`), not the chat transcript — which is how deep agents keep long artifacts out
of the context window. Durable, cross-session memory is different: it lives under `/memories/`,
persisted to the **Oracle AI Database** (Part 5.2).

In [ ]:
show_files(state)

FILE /research_request.md  (506 chars)


# Research Request

**Date:** Current session

**Requested by:** Oncology Tumor Board

**Question:**
Produce a one-page evidence brief for an oncology tumor board on what predicts survival and recurrence across cancers. Cover:
1. The strongest prognostic factors
2. How treatment choice affects outcomes
3. Where the evidence is mixed or uncertain
4. A practical recommendation for follow-up

Research each angle against the PubMed corpus, verify the draft with the critic, and cite PubMed IDs throughout.


FILE /final_report.md  (8678 chars)


# Predicting Survival & Recurrence Across Cancers
## Evidence Brief for Oncology Tumor Board
*All claims grounded in PubMed corpus; verified by independent critic review.*

---

## 1. STRONGEST PROGNOSTIC FACTORS

**TNM Stage** is the most reproducible pan-cancer prognostic determinant. In nasopharyngeal carcinoma (NPC), T4 disease carried a 31% local failure rate vs. 13% for T1–T3 (p=0.007) [PubMed:8985020]. In HCC, ultrasonographic surveillance — which detects disease at earlier stages — was associated with adjusted ORs for death of 0.58, 0.45, and 0.44 at 1, 2, and 3 years vs. unscreened patients [PubMed:15530261].

**Lymph Node Involvement** is a strong, independent predictor across solid tumors. In urothelial carcinoma, nodal status outperformed HER2 immunoreactivity as an independent prognostic indicator on multivariate Cox analysis [PubMed:17940352]. In pT3 renal cell carcinoma, lymph node involvement was an independent predictor of survival alongside VEGF expression (OR 6.07) and distant metastases [PubMed:17489316]. In rectal cancer, lymph node yield ≥12 was associated with better outcomes; low-volume centers had lower odds of adequate nodal harvest (OR=0.51) and worse 5-year survival [PubMed:29112560].

**Performance Status** is an independent prognostic factor in advanced NSCLC: ECOG 0–1 vs. 2 carried HR=1.63 (95% CI 1.04–2.54, p=0.032) for OS in a prospective cohort of 391 patients. Patient-reported pain (HR=1.11 per 10-point worsening, p<0.001) and dysphagia (HR=1.12, p=0.003) were also independent predictors [PubMed:16968876].

**Metastatic Burden** is a quantitative prognostic variable in prostate cancer patients who develop metastases after curative-intent radiotherapy: those with ≤5 bone lesions had 5- and 10-year OS of 73%/36% vs. 45%/18% for >5 lesions (p=0.02) [PubMed:14697414]. PSA level, Gleason grade, and pathological stage form the validated triad for predicting biochemical recurrence and prostate cancer-specific mortality after radical prostatectomy [PubMed:23806388].

**Molecular Markers**: VEGF overexpression was an independent predictor of survival in pT3 RCC (OR=6.07) [PubMed:17489316]. In GIST, KIT exon 11 mutations predicted significantly better response to neoadjuvant imatinib (84% vs. 40% for non-exon 11 mutants, p=0.01) [PubMed:27217036]. HER2 positivity in urothelial carcinoma was prognostically significant on univariate analysis (PFS p=0.02, OS p=0.005) but lost significance on multivariate analysis when nodal status was included [PubMed:17940352].

---

## 2. HOW TREATMENT CHOICE AFFECTS OUTCOMES

**Neoadjuvant chemotherapy + radiation vs. radiation alone** in Stage IV NPC: 5-year OS 68% vs. 56% (p=0.02), freedom from relapse 64% vs. 37% (p=0.01), local control 86% vs. 56% (p=0.009) [PubMed:8985020].

**Neoadjuvant targeted therapy enabling surgery** in GIST: patients who underwent surgical resection after imatinib had significantly improved event-free survival (p<0.001) and overall survival (p=0.021) vs. those who did not proceed to surgery — a mutation-agnostic effect [PubMed:27217036].

**Surgical volume and quality** in rectal cancer: patients treated at low-volume centers had a 34% higher risk of 5-year mortality, lower rates of adequate lymph node harvest (OR=0.51), lower receipt of neoadjuvant chemoradiation (OR=0.67), and higher 30-day (OR=3.38) and 90-day mortality (OR=2.07) [PubMed:29112560].

**Radiation sequencing trade-off** in rectal cancer: prior radiotherapy for the primary tumor was associated with lower R0 resection rates at salvage pelvic exenteration (63% vs. 87%, p=0.010), higher complication rates (p=0.014), and worse DFS (p=0.022) and OS (p=0.049) — effects that persisted on multivariate analysis adjusting for T and N stage [PubMed:25489696].

**FDG-PET-guided management** in rectal cancer: PET altered staging in 39% of patients and changed clinical management in 17%, including cancellation of surgery in 13% [PubMed:14978612].

---

## 3. MIXED OR UNCERTAIN EVIDENCE

**PSA as a surrogate endpoint**: Four validated postoperative nomograms predicted prostate cancer-specific mortality with *higher* c-index values than they predicted biochemical recurrence (BCR) itself, exposing a fundamental disconnect between the surrogate endpoint (PSA rise) and the clinically meaningful outcome (death) [PubMed:23806388]. Among patients who developed bone metastases after radiotherapy, the interval from metastasis diagnosis to death was not significantly different between the ≤5 and >5 lesion groups (p=0.17), challenging whether oligometastatic status reflects a biologically distinct disease course [PubMed:14697414].

**Axillary lymph node assessment in breast cancer**: Clinical examination of the axilla was inaccurate in 41% of cases overall; the false-positive rate was 53% for moderately suspicious nodes and 23% for highly suspicious nodes — meaning a substantial proportion of patients may be referred for full ALND based on unreliable clinical assessment [PubMed:15631914].

**HER2 as a prognostic biomarker in urothelial carcinoma**: Significant on univariate analysis but not on multivariate analysis — a classic pattern of confounded biomarker studies that explains why HER2 has not been validated as an independent prognostic marker in bladder cancer [PubMed:17940352].

**Histological grade in HCC**: Well-differentiated (Grade I) small HCCs showed no significant difference in disease-free survival vs. less-differentiated tumors, and lower rates of fibrous capsule formation, suggesting grade alone is insufficient as a recurrence risk stratifier [PubMed:8847047].

**Leptin/obesity paradox**: In advanced lung cancer, serum leptin was significantly lower in cancer patients than BMI-matched controls (4.75 vs. 9.67 ng/mL, p<0.001), yet showed no association with OS, challenging the assumption that leptin mediates the obesity–cancer outcome relationship [PubMed:28127977].

**Surveillance bias**: In endometrial cancer, asymptomatic recurrences detected through planned follow-up had dramatically better post-relapse survival (35 vs. 13 months, p=0.0001), but lead-time and length-time bias cannot be excluded from this retrospective design [PubMed:22706226].

---

## 4. PRACTICAL RECOMMENDATIONS FOR FOLLOW-UP

1. **Structured imaging-based surveillance is superior to clinical examination alone.** In endometrial cancer, imaging was the first diagnostic trigger in 62.4% of asymptomatic recurrences, and asymptomatic detection was associated with 35-month vs. 13-month post-relapse survival [PubMed:22706226]. In HCC, ultrasonographic surveillance was associated with adjusted ORs for death of 0.44–0.58 vs. unscreened patients [PubMed:15530261].

2. **Consider FDG-PET for recurrence detection in cervical cancer.** FDG-PET maintained high diagnostic accuracy for local recurrence regardless of glycemic status; in diabetic patients, it demonstrated superior metastatic lesion detection vs. CT/MRI (AUC 0.956 vs. 0.824, p=0.012) [PubMed:15703931]. Note: this study's primary aim was to assess whether diabetes impairs PET accuracy; a direct head-to-head PET vs. CT/MRI comparison in non-diabetic patients is not available from this corpus.

3. **In prostate cancer, track PSA doubling time, not PSA rise alone.** Nomogram validation data identify PSADT <9 months as the threshold defining aggressive biochemical recurrence with the highest risk of cancer-specific death [PubMed:23806388]. This threshold is used in nomogram validation; its direct use as a clinical treatment trigger should be interpreted in the context of individual patient risk.

4. **Refer rectal cancer patients to high-volume centers** for both primary surgery and salvage procedures. Low-volume centers are independently associated with worse survival, lower nodal harvest, and higher perioperative mortality [PubMed:29112560]. Prior radiotherapy narrows the window of curability at recurrence [PubMed:25489696].

5. **Maintain intensive post-resection surveillance in colorectal cancer with synchronous liver metastases.** Five-year disease-free survival is only 11–14% after resection, underscoring the high relapse burden and the need for surveillance capable of identifying re-resectable recurrence [PubMed:22537902].

---

### Corpus Limitations
This brief is grounded exclusively in the available PubMed oncology corpus. Topics not represented — and for which no claims are made — include: checkpoint immunotherapy outcomes, EGFR/KRAS/MSI/TMB as prognostic markers, CEA-based colorectal surveillance RCTs, CA-125 ovarian monitoring, and low-dose CT lung follow-up. Clinicians should supplement with current ASCO/ESMO/NCCN guidelines for these areas.


## 4.4 · Power #3 — Sub-agents (delegation & context isolation)

Each sub-topic was delegated to a fresh `kb-researcher`; the draft was checked by the `critic`. The raw retrieved abstracts stayed inside those sub-agents.

In [ ]:
show_delegations(state)

5 delegation(s) via the built-in `task` tool:
  -> [kb-researcher] Research the strongest prognostic factors that predict survival and recurrence across mult
  -> [kb-researcher] Research how treatment choice affects survival and recurrence outcomes across cancers. Foc
  -> [kb-researcher] Research areas where the evidence is mixed, uncertain, or actively debated regarding cance
  -> [kb-researcher] Research evidence-based recommendations for post-treatment follow-up and surveillance in c
  -> [critic] Please verify the following draft oncology evidence brief against the PubMed corpus. For e


## 4.5 · The final, verified brief

In [ ]:
display(Markdown(state["messages"][-1].content))

---

# Predicting Survival & Recurrence Across Cancers
## Evidence Brief for Oncology Tumor Board
*Critic-verified. All claims grounded in PubMed corpus with inline citations.*

---

## 1. STRONGEST PROGNOSTIC FACTORS

**TNM Stage** is the most reproducible pan-cancer prognostic determinant. In nasopharyngeal carcinoma (NPC), T4 disease carried a 31% local failure rate vs. 13% for T1–T3 (p=0.007) [PubMed:8985020]. In HCC, ultrasonographic surveillance — which detects disease at earlier stages — was associated with adjusted ORs for death of 0.58, 0.45, and 0.44 at 1, 2, and 3 years vs. unscreened patients [PubMed:15530261].

**Lymph Node Involvement** is a strong, independent predictor across solid tumors. In urothelial carcinoma, nodal status outperformed HER2 immunoreactivity as an independent prognostic indicator on multivariate Cox analysis [PubMed:17940352]. In pT3 renal cell carcinoma, lymph node involvement was an independent predictor of survival alongside VEGF expression (OR 6.07) and distant metastases [PubMed:17489316]. In rectal cancer, lymph node yield ≥12 was associated with better outcomes; low-volume centers had lower odds of adequate nodal harvest (OR=0.51) and worse 5-year survival [PubMed:29112560].

**Performance Status** is an independent prognostic factor in advanced NSCLC: ECOG 0–1 vs. 2 carried HR=1.63 (95% CI 1.04–2.54, p=0.032) for OS in a prospective cohort of 391 patients. Patient-reported pain (HR=1.11 per 10-point worsening, p<0.001) and dysphagia (HR=1.12, p=0.003) were also independent predictors [PubMed:16968876].

**Metastatic Burden** is a quantitative prognostic variable in prostate cancer patients who develop metastases after curative-intent radiotherapy: those with ≤5 bone lesions had 5- and 10-year OS of 73%/36% vs. 45%/18% for >5 lesions (p=0.02) [PubMed:14697414]. PSA level, Gleason grade, and pathological stage form the validated triad for predicting biochemical recurrence and prostate cancer-specific mortality after radical prostatectomy [PubMed:23806388].

**Molecular Markers**: VEGF overexpression was an independent predictor of survival in pT3 RCC (OR=6.07) [PubMed:17489316]. In GIST, KIT exon 11 mutations predicted significantly better response to neoadjuvant imatinib (84% vs. 40% for non-exon 11 mutants, p=0.01) [PubMed:27217036]. HER2 positivity in urothelial carcinoma was prognostically significant on univariate analysis (PFS p=0.02, OS p=0.005) but lost significance on multivariate analysis when nodal status was included [PubMed:17940352].

---

## 2. HOW TREATMENT CHOICE AFFECTS OUTCOMES

**Neoadjuvant chemotherapy + radiation vs. radiation alone** in Stage IV NPC: 5-year OS 68% vs. 56% (p=0.02), freedom from relapse 64% vs. 37% (p=0.01), local control 86% vs. 56% (p=0.009) [PubMed:8985020].

**Neoadjuvant targeted therapy enabling surgery** in GIST: patients who underwent surgical resection after imatinib had significantly improved event-free survival (p<0.001) and overall survival (p=0.021) vs. those who did not proceed to surgery — a mutation-agnostic effect [PubMed:27217036].

**Surgical volume and quality** in rectal cancer: patients treated at low-volume centers had a 34% higher risk of 5-year mortality, lower rates of adequate lymph node harvest (OR=0.51), lower receipt of neoadjuvant chemoradiation (OR=0.67), and higher 30-day (OR=3.38) and 90-day mortality (OR=2.07) [PubMed:29112560].

**Radiation sequencing trade-off** in rectal cancer: prior radiotherapy for the primary tumor was associated with lower R0 resection rates at salvage pelvic exenteration (63% vs. 87%, p=0.010), higher complication rates (p=0.014), and worse DFS (p=0.022) and OS (p=0.049) — effects that persisted on multivariate analysis adjusting for T and N stage [PubMed:25489696].

**FDG-PET-guided management** in rectal cancer: PET altered staging in 39% of patients and changed clinical management in 17%, including cancellation of surgery in 13% [PubMed:14978612].

---

## 3. MIXED OR UNCERTAIN EVIDENCE

**PSA as a surrogate endpoint**: Four validated postoperative nomograms predicted prostate cancer-specific mortality with *higher* c-index values than they predicted BCR itself, exposing a fundamental disconnect between the surrogate endpoint (PSA rise) and the clinically meaningful outcome (death) [PubMed:23806388]. Among patients who developed bone metastases after radiotherapy, the interval from metastasis diagnosis to death was not significantly different between the ≤5 and >5 lesion groups (p=0.17), challenging whether oligometastatic status reflects a biologically distinct disease course vs. earlier detection [PubMed:14697414].

**Axillary lymph node assessment in breast cancer**: Clinical examination of the axilla was inaccurate in 41% of cases overall; the false-positive rate was 53% for moderately suspicious nodes and 23% for highly suspicious nodes — meaning a substantial proportion of patients may be referred for full ALND based on unreliable clinical assessment [PubMed:15631914].

**HER2 as a prognostic biomarker in urothelial carcinoma**: Significant on univariate analysis but not on multivariate analysis — a classic pattern of confounded biomarker studies that explains why HER2 has not been validated as an independent prognostic marker in bladder cancer despite decades of investigation [PubMed:17940352].

**Histological grade in HCC**: Well-differentiated (Grade I) small HCCs showed no significant difference in disease-free survival vs. less-differentiated tumors, and lower rates of fibrous capsule formation, suggesting grade alone is insufficient as a recurrence risk stratifier [PubMed:8847047].

**Leptin/obesity paradox**: In advanced lung cancer, serum leptin was significantly lower in cancer patients than BMI-matched controls (4.75 vs. 9.67 ng/mL, p<0.001), yet showed no association with OS, challenging the assumption that leptin mediates the obesity–cancer outcome relationship [PubMed:28127977].

**Surveillance bias**: In endometrial cancer, asymptomatic recurrences detected through planned follow-up had dramatically better post-relapse survival (35 vs. 13 months, p=0.0001), but lead-time and length-time bias cannot be excluded from this retrospective design [PubMed:22706226].

---

## 4. PRACTICAL RECOMMENDATIONS FOR FOLLOW-UP

| # | Recommendation | Key Evidence |
|---|---|---|
| 1 | **Use structured imaging-based surveillance, not clinical exam alone.** In endometrial cancer, imaging triggered 62.4% of asymptomatic recurrence detections; asymptomatic detection was associated with 35- vs. 13-month post-relapse survival. | [PubMed:22706226]; [PubMed:15530261] |
| 2 | **Consider FDG-PET for recurrence detection in cervical cancer.** PET maintained high accuracy for local recurrence regardless of glycemic status; in diabetic patients, it outperformed CT/MRI for metastatic lesion detection (AUC 0.956 vs. 0.824, p=0.012). *Caveat: direct PET vs. CT/MRI comparison in non-diabetic patients is not available from this corpus.* | [PubMed:15703931] |
| 3 | **In prostate cancer, track PSA doubling time, not PSA rise alone.** PSADT <9 months defines aggressive BCR with the highest risk of cancer-specific death in nomogram validation data. *This threshold is a validated risk-stratifier; its use as a direct treatment trigger requires clinical judgment.* | [PubMed:23806388] |
| 4 | **Refer rectal cancer patients to high-volume centers** for primary surgery and salvage procedures. Low-volume centers are independently associated with worse survival, lower nodal harvest, and higher perioperative mortality. Prior radiotherapy narrows the window of curability at recurrence. | [PubMed:29112560]; [PubMed:25489696] |
| 5 | **Maintain intensive post-resection surveillance in colorectal cancer with synchronous liver metastases.** Five-year DFS is only 11–14% after resection, underscoring the high relapse burden and the need for surveillance capable of identifying re-resectable recurrence. | [PubMed:22537902] |

---

### ⚠️ Corpus Limitations
This brief is grounded exclusively in the available PubMed oncology corpus. Topics not represented — and for which **no claims are made** — include: checkpoint immunotherapy (PD-1/PD-L1) outcomes, EGFR/KRAS/MSI/TMB as prognostic markers, CEA-based colorectal surveillance RCTs, CA-125 ovarian monitoring, and low-dose CT lung follow-up after resection. Clinicians should supplement with current **ASCO/ESMO/NCCN guidelines** for these areas.

---
*Brief saved to `/final_report.md`. Critic review completed; 2 critical errors and 5 moderate/minor issues corrected before final publication.*

# Part 5 · Stream the loop, persist memory, and go further

## 5.1 · Power #4 — Stream the plan → act → observe loop

`invoke` waits for the final state. `stream(..., stream_mode="updates")` yields one update per graph
step, so you can watch the agent think, call tools, and observe results in real time. For a crisp
demo we build a **leaner** agent from the same factory with **no sub-agents** — so it plans and
retrieves directly.

In [ ]:
quick_agent = create_deepagents_agent(
    model=llm,
    datastores={"pubmed": store},      # one line -> Oracle search/get_document/stats + deep harness
    embedding_model=embedding_model,
    tools=[think_tool],                # no subagents -> it searches directly
    system_prompt=(
        "You are a biomedical research assistant. Search the corpus, then answer concisely and cite "
        "PubMed IDs like [PubMed:25957366]. If the corpus lacks an answer, say so."
    ),
)

print("Watching the agent work, step by step:\n")
for chunk in quick_agent.stream(
    {"messages": [HumanMessage(content="In two sentences, what does the corpus say about radiotherapy and local recurrence?")]},
    stream_mode="updates",
):
    for node, update in chunk.items():
        msgs = update.get("messages") if isinstance(update, dict) else None
        last = msgs[-1] if msgs else None
        kind = type(last).__name__ if last is not None else "-"
        note = ""
        if last is not None and getattr(last, "tool_calls", None):
            note = "  (calls: " + ", ".join(tc["name"] for tc in last.tool_calls) + ")"
        print(f"[{node}] {kind}{note}")

Watching the agent work, step by step:

[PatchToolCallsMiddleware.before_agent] -


[model] AIMessage  (calls: search)
[TodoListMiddleware.after_model] -
[tools] ToolMessage


[model] AIMessage
[TodoListMiddleware.after_model] -


## 5.2 · Power #5 — Durable memory in Oracle (checkpoints + long-term store)

Two kinds of persistence, both in Oracle:

- **Checkpointing** — because we passed `OracleSaver`, the research thread above was saved step by
  step. Re-invoking with the **same `thread_id`** resumes from Oracle instead of starting over.
- **Long-term memory** — the agent writes durable facts to `/memories/` (routed to `OracleStore`),
  which persist across *different* threads and even fresh processes.

**(a) Checkpoint resume** — the same `thread_id` picks up the Oracle-persisted conversation:

In [ ]:
resumed = agent.invoke(
    {"messages": [HumanMessage(content="Which sub-topics did you research for that brief? Just list them.")]},
    config=RESEARCH_CONFIG,   # same thread_id -> resumes from the OracleSaver checkpoint
)
print(f"Resumed the thread from Oracle — it now carries {len(resumed['messages'])} messages.\n")
display(Markdown(resumed["messages"][-1].content))

Resumed the thread from Oracle — it now carries 25 messages.



1. Strongest prognostic factors for survival and recurrence across cancers
2. How treatment choice affects survival and recurrence outcomes
3. Mixed or uncertain evidence in cancer prognosis and recurrence prediction
4. Evidence-based recommendations for post-treatment follow-up

**(b) Long-term memory** — save a durable preference, then recall it on a brand-new thread. It physically lives in Oracle (via `OracleStore`):

In [ ]:
# Save a durable preference.
agent.invoke(
    {"messages": [HumanMessage(content=(
        "Remember for future briefs: our tumor board wants recommendations capped at 3 bullets. "
        "Save that to /memories/board_prefs.md."))]},
    config={"configurable": {"thread_id": "prefs-writer"}},
)

# It physically persisted to Oracle (OracleStore):
print("What is in the Oracle long-term store:")
for item in memory_store.search(("oncology", "memories")):
    print(f"  {item.key}: {str(item.value)[:90]}")

What is in the Oracle long-term store:
  /board_prefs.md: {'content': '# Tumor Board Preferences\n\n- **Recommendations**: Cap at 3 bullets per brie


One saved preference found:

- **Recommendations**: Cap at 3 bullets per brief.

In [ ]:
# A fresh, unrelated session reads it back from Oracle:
recall = agent.invoke(
    {"messages": [HumanMessage(content="Any saved preferences for our briefs? Check /memories/.")]},
    config={"configurable": {"thread_id": "a-totally-new-session"}},
)
print()
display(Markdown(recall["messages"][-1].content))

## 5.3 · A note on filesystem backends — why `StoreBackend`, and do we need `StateBackend`?

A natural question: should the agent's files live in Oracle's **Database File System (DBFS)**? DBFS
stores files as SecureFiles LOBs behind a POSIX-like interface via `DBMS_FS` —
[Oracle docs](https://docs.oracle.com/en/database/oracle/oracle-database/21/adlob/what-is-database-file-system.html).
It's a great fit for *mounting* a filesystem onto the database, but it's the wrong layer here:
deepagents' filesystem is a Python `BackendProtocol`, and we already persist agent files to Oracle
the idiomatic LangGraph way — **`StoreBackend` over `OracleStore`** (that's what `/memories/` uses).
Wiring DBFS would mean a custom backend over `DBMS_FS` that duplicates `StoreBackend` with far more
setup.

And we keep **`StateBackend`** on purpose: it's the *fast, ephemeral* scratch layer (no DB
round-trip for every transient read/write while the agent drafts). The deepagents-recommended
pattern is exactly the `CompositeBackend(StateBackend + StoreBackend@/memories/)` we built — fast
scratch where durability doesn't matter, Oracle-backed persistence where it does.

## 5.4 · Cleanup

In [ ]:
store.close()
for _c in (saver_conn, store_conn):
    try:
        _c.close()
    except Exception:
        pass
print("✓ Datastore + Oracle persistence connections closed.")

✓ Datastore + Oracle persistence connections closed.


## 5.5 · Recap & key takeaways

- **Deep research over a real, private store is the canonical deep-agent use case.** We exercised
  all five powers — **planning**, a **virtual filesystem**, **sub-agents**, **streaming**, and
  **durable memory** — on a broad clinical question over **real PubMed abstracts**, and got a
  structured, verified, PubMed-ID-cited brief.
- **One Oracle AI Database for the whole agent stack.** `OracleVS` (`langchain-oracledb`) with an
  **HNSW** index grounds retrieval, `OracleSaver` checkpoints every thread, and `OracleStore` backs
  the `/memories/` long-term store — retrieval, working state, and durable memory in one system of
  record, behind one connection pool.
- **Hybrid retrieval in the database.** Co-locating the `VECTOR` embedding (HNSW) with the text and
  an Oracle Text index lets one tool fuse semantic + keyword search — without a separate vector DB.
- **Bring-your-own-model decouples the brain from the data.** `create_deepagents_agent(model=...)`
  ran the Oracle harness on **Claude** with zero OCI credentials; the data stayed local in Oracle.

### Variations to try
- **Go fully local:** swap `ChatAnthropic` for a local Ollama model
  (`ChatOllama(model="qwen3", temperature=0)`) for a zero-cloud stack. Use a strong tool-caller.
- **Other providers:** `model=ChatOpenAI(model="gpt-5")`, a self-hosted vLLM endpoint, or a
  custom-configured `ChatOCIGenAI`. The agent code is identical.
- **Scale the corpus:** lift the `len(CORPUS) >= 40` cap or drop the oncology filter to index more
  of PubMedQA — the **HNSW** index is what keeps search fast as the corpus grows.
- **Scope memory per user:** namespace the `/memories/` `OracleStore` by user or org via the
  `namespace=` callback for multi-tenant long-term memory.

> **🔑 Final takeaway:** A deep agent is a small set of powers — *plan, write to files, delegate to
> sub-agents, stream, and remember* — wrapped around an LLM. Point those powers at the **Oracle AI
> Database** for grounding (HNSW), memory, and checkpoints, and a **Claude** brain for reasoning,
> and you have a production-shaped research analyst you can trust — because every claim is cited to a
> real paper and verified, and every session is durable.